# Denoising Process Visualization (GSM8K)

在 GSM8K 上跑 50 个 task，收集每步解码快照，生成一个 HTML 文件供离线浏览。

## 1. 环境设置

In [ ]:
import os, sys, gc
import torch
import torch.nn.functional as F
import numpy as np

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
os.chdir(os.path.join(NOTEBOOK_DIR, 'llada'))
print(f'Working dir: {os.getcwd()}')

# viz_static.py is in the parent dir
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

torch.cuda.empty_cache(); gc.collect()
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

## 2. 模型 & 数据加载

In [ ]:
from transformers import AutoTokenizer, AutoConfig
from model.modeling_llada import LLaDAModelLM
from datasets import load_dataset

MODEL_PATH = 'GSAI-ML/LLaDA-8B-Instruct'
MASK_ID = 126336

config = AutoConfig.from_pretrained(MODEL_PATH)
config.flash_attention = True
model = LLaDAModelLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16, config=config,
).eval().to('cuda')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

gsm8k = load_dataset('gsm8k', 'main', split='test')
print(f'Model loaded. GSM8K test: {len(gsm8k)} samples')

In [ ]:
FEW_SHOT_EXAMPLES = """Question: Jen and Tyler are gymnasts practicing flips. Jen is practicing the triple-flip while Tyler is practicing the double-flip. Jen did sixteen triple-flips during practice. Tyler flipped in the air half the number of times Jen did. How many double-flips did Tyler do?
Answer: Jen did 16 triple-flips, so she did 16 * 3 = <<16*3=48>>48 flips.
Tyler did half the number of flips, so he did 48 / 2 = <<48/2=24>>24 flips.
A double flip has two flips, so Tyler did 24 / 2 = <<24/2=12>>12 double-flips.
#### 12

Question: Four people in a law firm are planning a party. Mary will buy a platter of pasta for $20 and a loaf of bread for $2. Elle and Andrea will split the cost for buying 4 cans of soda which cost $1.50 each, and chicken wings for $10. Joe will buy a cake that costs $5. How much more will Mary spend than the rest of the firm put together?
Answer: Mary will spend $20 + $2 = $<<20+2=22>>22.
Elle and Andrea will spend $1.5 x 4 = $<<1.5*4=6>>6 for the soda.
Elle and Andrea will spend $6 + $10 = $<<6+10=16>>16 for the soda and chicken wings.
Elle, Andrea, and Joe together will spend $16 + $5 = $<<16+5=21>>21.
So, Mary will spend $22 - $21 = $<<22-21=1>>1 more than all of them combined.
#### 1

Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?
Answer: The grill burned 3 * 60 = <<3*60=180>>180 coals.
It takes 20 minutes to burn 15 coals, so the grill ran for 180 / 15 * 20 = <<180/15*20=240>>240 minutes.
#### 240

Question: A bear is preparing to hibernate for the winter and needs to gain 1000 pounds. At the end of summer, the bear feasts on berries and small woodland animals. During autumn, it devours acorns and salmon. It gained a fifth of the weight it needed from berries during summer, and during autumn, it gained twice that amount from acorns. Salmon made up half of the remaining weight it had needed to gain. How many pounds did it gain eating small animals?
Answer: The bear gained 1 / 5 * 1000 = <<1/5*1000=200>>200 pounds from berries.
It gained 2 * 200 = <<2*200=400>>400 pounds from acorns.
It still needed 1000 - 200 - 400 = <<1000-200-400=400>>400 pounds.
Thus, it gained 400 / 2 = <<400/2=200>>200 pounds from salmon.
Therefore, the bear gained 400 - 200 = <<400-200=200>>200 pounds from small animals.
#### 200

Question: Brendan can cut 8 yards of grass per day, he bought a lawnmower and it helped him to cut more yards by Fifty percent per day. How many yards will Brendan be able to cut after a week?
Answer: The additional yard Brendan can cut after buying the lawnmower is 8 x 0.50 = <<8*0.50=4>>4 yards.
So, the total yards he can cut with the lawnmower is 8 + 4 = <<8+4=12>>12.
Therefore, the total number of yards he can cut in a week is 12 x 7 = <<12*7=84>>84 yards.
#### 84"""


def build_prompt(question: str) -> torch.Tensor:
    text = FEW_SHOT_EXAMPLES + f'\n\nQuestion: {question}\nAnswer:'
    messages = [{'role': 'user', 'content': text}]
    formatted = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    ids = tokenizer(formatted)['input_ids']
    return torch.tensor(ids, dtype=torch.long, device='cuda').unsqueeze(0)


import re

def extract_answer(text: str) -> str | None:
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    return m.group(1).replace(',', '').strip() if m else None


LIMIT = 50
prompts = [build_prompt(gsm8k[i]['question']) for i in range(LIMIT)]
questions = [gsm8k[i]['question'] for i in range(LIMIT)]
ref_answers = [gsm8k[i]['answer'] for i in range(LIMIT)]
print(f'Built {len(prompts)} prompts, first length: {prompts[0].shape[1]} tokens')

## 3. 跑 50 个 task + 收集 denoising 数据

In [ ]:
from viz_static import generate_with_collection, process_sample
import time

GEN_LENGTH = 256
STEPS = 256
BLOCK_LENGTH = 32
THRESHOLD = 0.9

all_samples_json = []

t0 = time.time()
for i in range(LIMIT):
    prompt = prompts[i]

    x, nfe, history = generate_with_collection(
        model, prompt,
        steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
        temperature=0.0, threshold=THRESHOLD, mask_id=MASK_ID,
    )

    # Decode output
    gen_text = tokenizer.decode(x[0, prompt.shape[1]:], skip_special_tokens=True)
    for stop in ['Question:', '\n\nQuestion']:
        if stop in gen_text:
            gen_text = gen_text.split(stop)[0]
    gen_ans = extract_answer(gen_text)
    ref_ans = extract_answer(ref_answers[i])
    correct = gen_ans is not None and ref_ans is not None and gen_ans == ref_ans

    # Convert to JSON-ready format
    sample_json = process_sample(
        history, tokenizer,
        question=questions[i],
        gen_answer=f"{gen_ans} {'✓' if correct else '✗ (ref: '+str(ref_ans)+')'}",
        ref_answer=ref_ans,
    )
    all_samples_json.append(sample_json)

    elapsed = time.time() - t0
    print(f'[{i+1}/{LIMIT}] NFE={nfe:3d} ans={gen_ans} {"✓" if correct else "✗"} ({elapsed:.1f}s)')

correct_count = sum(1 for s in all_samples_json if '✓' in s.get('gen_answer', ''))
print(f'\nDone! {correct_count}/{LIMIT} correct. Total time: {time.time()-t0:.1f}s')

## 4. 生成 HTML

In [ ]:
from viz_static import generate_multi_html

html = generate_multi_html(all_samples_json, title=f'GSM8K Denoising — {LIMIT} samples')

output_path = os.path.join(NOTEBOOK_DIR, 'viz_gsm8k_50.html')
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html)

size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f'✓ HTML saved: {output_path}')
print(f'  Size: {size_mb:.1f} MB')
print(f'  下载到本地用浏览器打开即可浏览全部 {LIMIT} 个 sample 的 denoising 过程')

In [ ]:
# (可选) 在 notebook 内预览
from IPython.display import IFrame
IFrame(src=output_path, width='100%', height=700)